# EXP21D-v4 FairCtrl-1A
Fair comparison of supervised 128-d Tabular MLP and GraphSAGE embeddings with the same XGBoost classifier. The notebook persists embeddings, predictions, histories, models, and paper-ready figures.


In [ ]:
%%writefile requirements.txt
numpy>=1.26
pandas>=2.1
scikit-learn>=1.4
matplotlib>=3.8
PyYAML>=6.0
torch>=2.2
xgboost>=2.0


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP21D"
  version: "v4_fairctrl_1a"
  name: "Fair 128-d Neural Embedding Control"
  seed: 42
  evaluation_seed: 1042
data:
  transactions: "/kaggle/input/ibm-credit-card-transactions/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016
time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60
features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true
model:
  input_dim_expected: 10
  hidden_channels: 128
  dropout: 0.2
training:
  batch_size: 4096
  inference_batch_size: 16384
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  grad_clip_norm: 5.0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
classifier:
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"
  n_jobs: -1
threshold_calibration:
  min_precision_constraint: 0.15
baseline:
  exp21d_v1_result_dir: "/kaggle/input/exp21d-v1-results"
  samplers: ["uniform", "topology", "importance"]
reproducibility:
  deterministic: true
  warn_only: true
  compute_dataset_sha256: false
smoke:
  max_rows_per_split: 20000
  epochs: 2
  xgb_n_estimators: 20
visualization:
  dpi: 300
  tsne_max_points: 5000
output:
  result_dir: "/kaggle/working/result/exp21d_v4_fairctrl_1a"
  snapshot_source: true
  save_embeddings: true
  save_tabular_features: true


In [ ]:
%%writefile config.py
from __future__ import annotations

from pathlib import Path

import yaml


def load_config(path: str | Path) -> dict:
    config_path = Path(path)
    if not config_path.exists():
        raise FileNotFoundError(f"Config tidak ditemukan: {config_path}")
    with config_path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    required = {"experiment", "data", "time_window", "features", "model", "training", "classifier", "output"}
    missing = sorted(required - set(config))
    if missing:
        raise ValueError(f"Config belum lengkap, bagian yang hilang: {missing}")
    return config


In [ ]:
%%writefile artifacts.py
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def write_json(path: str | Path, value: Any) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(target)


class RunLogger:
    """Append-only event history; a crashed run remains auditable."""

    def __init__(self, path: str | Path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def log(self, event: str, **payload: Any) -> None:
        record = {"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}
        with self.path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")


def build_artifact_index(run_dir: str | Path) -> dict[str, Any]:
    root = Path(run_dir)
    entries = []
    for item in sorted(root.rglob("*")):
        if item.is_file() and item.name != "artifact_index.json":
            entries.append({"path": item.relative_to(root).as_posix(), "size_bytes": item.stat().st_size})
    result = {"run_dir": str(root.resolve()), "file_count": len(entries), "files": entries}
    write_json(root / "artifact_index.json", result)
    return result


In [ ]:
%%writefile reproducibility.py
from __future__ import annotations

import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np


def stable_hash(value: Any, length: int = 12) -> str:
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:length]


def seed_everything(seed: int, deterministic: bool = True, warn_only: bool = True) -> None:
    import torch

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.use_deterministic_algorithms(True, warn_only=warn_only)


def file_fingerprint(path: str | Path, with_sha256: bool = False) -> dict[str, Any]:
    source = Path(path).resolve()
    stat = source.stat()
    result: dict[str, Any] = {"path": str(source), "size_bytes": stat.st_size, "modified_time_ns": stat.st_mtime_ns}
    if with_sha256:
        digest = hashlib.sha256()
        with source.open("rb") as handle:
            while block := handle.read(8 * 1024 * 1024):
                digest.update(block)
        result["sha256"] = digest.hexdigest()
    return result


def git_revision(path: str | Path) -> dict[str, Any]:
    try:
        commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=path, text=True, stderr=subprocess.DEVNULL).strip()
        dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=path, text=True).strip())
        return {"commit": commit, "dirty": dirty}
    except (OSError, subprocess.CalledProcessError):
        return {"commit": None, "dirty": None}


def environment_manifest() -> dict[str, Any]:
    import pandas
    import sklearn
    import torch
    import xgboost

    return {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pandas.__version__,
        "scikit_learn": sklearn.__version__,
        "torch": torch.__version__,
        "xgboost": xgboost.__version__,
        "cuda_runtime": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


def snapshot_source(source_dir: str | Path, destination: str | Path) -> None:
    source, target = Path(source_dir), Path(destination)
    target.mkdir(parents=True, exist_ok=True)
    for item in source.iterdir():
        if item.is_file() and item.suffix.lower() in {".py", ".yaml", ".yml", ".md", ".txt"}:
            shutil.copy2(item, target / item.name)


In [ ]:
%%writefile data_preprocessing.py
from __future__ import annotations

from datetime import timedelta

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


def load_and_split_raw_data(
    file_path: str,
    start_year: int = 2016,
    nrows: int | None = None,
    reference_date: str = "2018-01-01",
    train_days: int = 300,
    val_days: int = 30,
    gap_days: int = 8,
    test_days: int = 60,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """Load IBM transactions and create the locked 300/30/8/60 temporal split."""
    chunks: list[pd.DataFrame] = []
    collected = 0
    for chunk in pd.read_csv(file_path, chunksize=1_000_000):
        if start_year is not None:
            chunk = chunk[chunk["Year"] >= start_year]
        if nrows is not None:
            chunk = chunk.iloc[: max(0, nrows - collected)]
        if len(chunk):
            chunks.append(chunk)
            collected += len(chunk)
        if nrows is not None and collected >= nrows:
            break
    if not chunks:
        raise ValueError("Tidak ada transaksi setelah filter data diterapkan")

    frame = pd.concat(chunks, ignore_index=True)
    frame["transaction_id"] = np.arange(len(frame), dtype=np.int64)
    frame["Time"] = frame["Time"].fillna("00:00")
    frame["datetime"] = pd.to_datetime(
        frame["Year"].astype(str)
        + "-"
        + frame["Month"].astype(str).str.zfill(2)
        + "-"
        + frame["Day"].astype(str).str.zfill(2)
        + " "
        + frame["Time"]
    )
    frame = frame.sort_values(["datetime", "transaction_id"]).reset_index(drop=True)
    frame["Amount"] = (
        frame["Amount"].astype(str).str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    )
    frame["Is Fraud?"] = frame["Is Fraud?"].map({"Yes": 1, "No": 0})
    if frame["Is Fraud?"].isna().any():
        raise ValueError("Label Is Fraud? memuat nilai selain Yes/No")

    start = pd.Timestamp(reference_date)
    val_start = start + timedelta(days=train_days)
    gap_start = val_start + timedelta(days=val_days)
    test_start = gap_start + timedelta(days=gap_days)
    test_end = test_start + timedelta(days=test_days)

    def take(lower: pd.Timestamp, upper: pd.Timestamp) -> pd.DataFrame:
        return frame[(frame["datetime"] >= lower) & (frame["datetime"] < upper)].copy().reset_index(drop=True)

    train = take(start, val_start)
    validation = take(val_start, gap_start)
    gap = take(gap_start, test_start)
    test = take(test_start, test_end)
    splits = {"train": train, "validation": validation, "gap": gap, "test": test}
    if any(len(value) == 0 for value in splits.values()):
        raise ValueError(f"Ada split temporal kosong: { {key: len(value) for key, value in splits.items()} }")

    statistics = {
        "protocol": "SHINE FRAUD II 300d/30d/8d/60d",
        "boundaries": {
            "train": [str(start), str(val_start)],
            "validation": [str(val_start), str(gap_start)],
            "embargo_gap": [str(gap_start), str(test_start)],
            "test": [str(test_start), str(test_end)],
        },
        "splits": {
            name: {
                "rows": int(len(value)),
                "fraud": int(value["Is Fraud?"].sum()) if name != "test" else None,
                "fraud_rate": float(value["Is Fraud?"].mean()) if name != "test" else None,
                "first_timestamp": str(value["datetime"].min()),
                "last_timestamp": str(value["datetime"].max()),
            }
            for name, value in splits.items()
        },
    }
    return train, validation, test, gap, statistics


def apply_strict_anti_leakage_preprocessing(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    test: pd.DataFrame,
    gap: pd.DataFrame,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame, list[str], dict]:
    """Fit every learned transform on train only and map unseen categories to zero."""
    if feature_mode not in {"clean", "clean_basic"}:
        raise ValueError("feature_mode harus clean atau clean_basic")
    frames = [train, validation, test, gap]
    categorical = ["Use Chip", "MCC", "Errors?"]
    if feature_mode == "clean":
        categorical.insert(1, "Merchant State")

    if use_amount_log:
        for frame in frames:
            frame["amount_log"] = np.log1p(np.maximum(frame["Amount"].to_numpy(), 0.0)).astype(np.float32)

    amount_scaler = StandardScaler().fit(train[["Amount"]])
    for frame in frames:
        frame["Amount"] = amount_scaler.transform(frame[["Amount"]]).astype(np.float32)
    log_scaler = None
    if use_amount_log:
        log_scaler = StandardScaler().fit(train[["amount_log"]])
        for frame in frames:
            frame["amount_log"] = log_scaler.transform(frame[["amount_log"]]).astype(np.float32)

    encoders: dict[str, dict[str, int]] = {}
    oov_counts: dict[str, dict[str, int]] = {}
    for column in categorical:
        for frame in frames:
            frame[column] = frame[column].fillna("<missing>").astype(str)
        mapping = {value: index + 1 for index, value in enumerate(sorted(train[column].unique()))}
        encoders[column] = mapping
        for frame in frames:
            frame[column] = frame[column].map(mapping).fillna(0).astype(np.int32)
        oov_counts[column] = {
            "validation": int((validation[column] == 0).sum()),
            "test": int((test[column] == 0).sum()),
            "gap": int((gap[column] == 0).sum()),
        }

    features = ["Amount"] + (["amount_log"] if use_amount_log else []) + categorical
    if use_cyclical_time:
        for frame in frames:
            hour = frame["datetime"].dt.hour + frame["datetime"].dt.minute / 60.0
            day = frame["datetime"].dt.dayofweek
            frame["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype(np.float32)
            frame["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype(np.float32)
            frame["dow_sin"] = np.sin(2 * np.pi * day / 7).astype(np.float32)
            frame["dow_cos"] = np.cos(2 * np.pi * day / 7).astype(np.float32)
        features += ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]

    forbidden = {"User", "Merchant Name", "Zip", "Merchant City", "Is Fraud?"}
    leaked = sorted(forbidden.intersection(features))
    if leaked:
        raise RuntimeError(f"Fitur terlarang masuk model: {leaked}")
    artifacts = {
        "features": features,
        "fit_scope": "train_only",
        "amount_scaler": {"mean": amount_scaler.mean_.tolist(), "scale": amount_scaler.scale_.tolist()},
        "amount_log_scaler": None if log_scaler is None else {"mean": log_scaler.mean_.tolist(), "scale": log_scaler.scale_.tolist()},
        "categorical_encoders": encoders,
        "oov_counts": oov_counts,
    }
    return train, validation, test, gap, features, artifacts


def smoke_subset(frame: pd.DataFrame, max_rows: int, seed: int) -> pd.DataFrame:
    """Small deterministic split that retains all positives up to one quarter of the limit."""
    if len(frame) <= max_rows:
        return frame.copy().reset_index(drop=True)
    positive = frame[frame["Is Fraud?"] == 1].iloc[: max_rows // 4]
    negative = frame[frame["Is Fraud?"] == 0].sample(max_rows - len(positive), random_state=seed)
    return pd.concat([positive, negative]).sort_values(["datetime", "transaction_id"]).reset_index(drop=True)


In [ ]:
%%writefile metrics.py
from __future__ import annotations

from typing import Any

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)


def find_optimal_threshold(probabilities: np.ndarray, labels: np.ndarray, min_precision: float = 0.15) -> tuple[float, dict[str, Any]]:
    precision, recall, thresholds = precision_recall_curve(labels, probabilities)
    if thresholds.size == 0:
        return 0.5, {"tau": 0.5, "val_f1": 0.0, "candidate_thresholds": 0}
    p, r = precision[:-1], recall[:-1]
    f1 = np.divide(2 * p * r, p + r, out=np.zeros_like(p), where=(p + r) > 0)
    eligible = p >= min_precision
    indices = np.flatnonzero(eligible) if eligible.any() else np.arange(len(thresholds))
    best = int(indices[np.argmax(f1[indices])])
    return float(thresholds[best]), {
        "tau": float(thresholds[best]),
        "val_f1": float(f1[best]),
        "val_precision": float(p[best]),
        "val_recall": float(r[best]),
        "candidate_thresholds": int(len(thresholds)),
        "min_precision_constraint": float(min_precision),
        "constraint_applied": bool(eligible.any()),
    }


def evaluate_predictions(probabilities: np.ndarray, labels: np.ndarray, threshold: float) -> dict[str, Any]:
    predictions = (probabilities >= threshold).astype(np.int8)
    default = (probabilities >= 0.5).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    recall = float(recall_score(labels, predictions, zero_division=0))
    precision = float(precision_score(labels, predictions, zero_division=0))
    specificity = float(tn / (tn + fp)) if tn + fp else 0.0
    total_positive = int(labels.sum())
    operational: dict[str, Any] = {}
    for requested_k in (100, 200, 300, 421, 500, 1000):
        k = min(requested_k, len(labels))
        top = np.argpartition(probabilities, len(probabilities) - k)[-k:]
        captured = int(labels[top].sum())
        operational[str(requested_k)] = {
            "actual_k": int(k),
            "true_positives": captured,
            "recall": float(captured / total_positive) if total_positive else 0.0,
            "precision": float(captured / k) if k else 0.0,
        }
    return {
        "metrics_optimal_threshold": {
            "threshold": float(threshold),
            "f1_score": float(f1_score(labels, predictions, zero_division=0)),
            "recall": recall,
            "precision": precision,
            "accuracy": float(accuracy_score(labels, predictions)),
            "gmean": float(np.sqrt(recall * specificity)),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        },
        "metrics_default_threshold_05": {
            "threshold": 0.5,
            "f1_score": float(f1_score(labels, default, zero_division=0)),
            "recall": float(recall_score(labels, default, zero_division=0)),
            "precision": float(precision_score(labels, default, zero_division=0)),
        },
        "metrics_threshold_independent": {
            "auprc": float(average_precision_score(labels, probabilities)),
            "roc_auc": float(roc_auc_score(labels, probabilities)),
            "recall_at_k": operational,
        },
    }


def threshold_curve(probabilities: np.ndarray, labels: np.ndarray, points: int = 181) -> dict[str, list[float]]:
    thresholds = np.linspace(0.05, 0.95, points)
    f1_values, recall_values, precision_values = [], [], []
    for threshold in thresholds:
        prediction = probabilities >= threshold
        f1_values.append(float(f1_score(labels, prediction, zero_division=0)))
        recall_values.append(float(recall_score(labels, prediction, zero_division=0)))
        precision_values.append(float(precision_score(labels, prediction, zero_division=0)))
    return {
        "threshold": thresholds.tolist(),
        "f1": f1_values,
        "recall": recall_values,
        "precision": precision_values,
    }


In [ ]:
%%writefile model.py
from __future__ import annotations

import torch
import torch.nn as nn


class TabularEmbeddingMLP(nn.Module):
    """Two hidden-layer supervised tabular encoder with a 128-d latent output."""

    def __init__(self, input_dim: int, hidden_channels: int = 128, dropout: float = 0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels),
            nn.BatchNorm1d(hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(hidden_channels, 1)

    def forward(self, features: torch.Tensor, return_embeddings: bool = False) -> torch.Tensor:
        embeddings = self.encoder(features)
        if return_embeddings:
            return embeddings
        return self.classifier(embeddings).squeeze(-1)


def weighted_bce(logits: torch.Tensor, targets: torch.Tensor, pos_weight: float) -> torch.Tensor:
    weight = torch.as_tensor(pos_weight, dtype=logits.dtype, device=logits.device)
    return nn.functional.binary_cross_entropy_with_logits(logits, targets, pos_weight=weight)


def unweighted_bce(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return nn.functional.binary_cross_entropy_with_logits(logits, targets)


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import gc
import json
import time
import traceback
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.optim as optim
import yaml
from sklearn.metrics import average_precision_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

from artifacts import RunLogger, build_artifact_index, write_json
from config import load_config
from data_preprocessing import apply_strict_anti_leakage_preprocessing, load_and_split_raw_data, smoke_subset
from metrics import evaluate_predictions, find_optimal_threshold, threshold_curve
from model import TabularEmbeddingMLP, unweighted_bce, weighted_bce
from reproducibility import environment_manifest, file_fingerprint, git_revision, seed_everything, snapshot_source, stable_hash


MODEL_SPECS = {
    "tabular_10d": "10-d Clean Tabular + XGBoost",
    "mlp_embedding_128d": "128-d MLP Embedding + XGBoost",
    "hybrid_tabular_mlp_138d": "10-d Tabular + 128-d MLP Embedding + XGBoost",
}


def make_loader(features: np.ndarray, labels: np.ndarray, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels.astype(np.float32, copy=False)))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(seed),
    )


def infer_head(model: TabularEmbeddingMLP, loader: DataLoader, device: torch.device) -> dict[str, Any]:
    model.eval()
    probabilities, labels = [], []
    loss_sum = 0.0
    count = 0
    with torch.no_grad():
        for features, targets in loader:
            features, targets = features.to(device), targets.to(device)
            logits = model(features)
            loss_sum += float(unweighted_bce(logits, targets).item()) * len(targets)
            count += len(targets)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(targets.cpu().numpy().astype(np.int8))
    return {
        "probabilities": np.concatenate(probabilities),
        "labels": np.concatenate(labels),
        "loss": loss_sum / max(count, 1),
    }


def extract_embeddings(
    model: TabularEmbeddingMLP,
    features: np.ndarray,
    labels: np.ndarray,
    transaction_ids: np.ndarray,
    output_dir: Path,
    split: str,
    batch_size: int,
    device: torch.device,
    hidden_channels: int,
    save_tabular: bool,
) -> dict[str, Path]:
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = {
        "embedding": output_dir / f"{split}_mlp_embeddings.npy",
        "labels": output_dir / f"{split}_labels.npy",
        "transaction_ids": output_dir / f"{split}_transaction_ids.npy",
    }
    if save_tabular:
        paths["tabular"] = output_dir / f"{split}_tabular_features.npy"
        np.save(paths["tabular"], features.astype(np.float32, copy=False))
    np.save(paths["labels"], labels.astype(np.int8, copy=False))
    np.save(paths["transaction_ids"], transaction_ids.astype(np.int64, copy=False))
    embeddings = np.lib.format.open_memmap(
        paths["embedding"], mode="w+", dtype=np.float32, shape=(len(features), hidden_channels)
    )
    loader = make_loader(features, labels, batch_size, False, 0)
    offset = 0
    model.eval()
    with torch.no_grad():
        for batch_features, _ in loader:
            output = model(batch_features.to(device), return_embeddings=True).cpu().numpy()
            embeddings[offset : offset + len(output)] = output
            offset += len(output)
    embeddings.flush()
    del embeddings
    return paths


def assemble_hybrid(tabular: np.ndarray, embedding_path: Path, output_path: Path) -> Path:
    embedding = np.load(embedding_path, mmap_mode="r")
    hybrid = np.lib.format.open_memmap(
        output_path, mode="w+", dtype=np.float32, shape=(len(tabular), tabular.shape[1] + embedding.shape[1])
    )
    block = 100_000
    for start in range(0, len(tabular), block):
        stop = min(start + block, len(tabular))
        hybrid[start:stop, : tabular.shape[1]] = tabular[start:stop]
        hybrid[start:stop, tabular.shape[1] :] = embedding[start:stop]
    hybrid.flush()
    del hybrid, embedding
    return output_path


def xgb_parameters(config: dict, class_ratio: float, seed: int, smoke: bool) -> dict[str, Any]:
    strategy = config.get("scale_pos_weight_strategy", "sqrt")
    scale = float(np.sqrt(class_ratio) if strategy == "sqrt" else class_ratio if strategy == "full" else 1.0)
    return {
        "n_estimators": int(config["n_estimators"] if not smoke else min(config["n_estimators"], 20)),
        "max_depth": int(config["max_depth"]),
        "learning_rate": float(config["learning_rate"]),
        "min_child_weight": float(config["min_child_weight"]),
        "subsample": float(config["subsample"]),
        "colsample_bytree": float(config["colsample_bytree"]),
        "scale_pos_weight": scale,
        "tree_method": config.get("tree_method", "hist"),
        "random_state": seed,
        "n_jobs": int(config.get("n_jobs", -1)),
        "eval_metric": "logloss",
    }


def load_graphsage_baselines(result_dir: Path, samplers: list[str]) -> list[dict[str, Any]]:
    rows = []
    for sampler in samplers:
        path = result_dir / f"exp21d_{sampler}_results.json"
        if not path.exists():
            raise FileNotFoundError(f"Baseline GraphSAGE tidak ditemukan: {path}")
        result = json.loads(path.read_text(encoding="utf-8"))
        ablations = {item["subset_name"]: item for item in result.get("feature_subsets_ablation", [])}
        graph = ablations.get("GraphSAGE Only")
        if graph is None:
            raise KeyError(f"GraphSAGE Only tidak ada di {path}")
        rows.append(
            {
                "representation": f"GraphSAGE-{sampler} embedding",
                "sampler": sampler,
                "dimensions": int(result.get("num_features_gnn", 128)),
                "downstream_classifier": "XGBoost",
                "test_auprc": float(graph["test_auprc"]),
                "test_roc_auc": float(graph["test_roc_auc"]),
                "test_f1": float(graph["test_f1"]),
                "test_recall": float(graph["test_recall"]),
                "test_precision": float(graph["test_precision"]),
                "optimal_threshold": float(graph["optimal_tau"]),
                "source": str(path.resolve()),
            }
        )
    return rows


def save_preprocessing_artifacts(path: Path, artifacts: dict[str, Any]) -> None:
    write_json(path, artifacts)


def run_experiment(args: argparse.Namespace) -> Path:
    config_path = Path(args.config).resolve()
    cfg = load_config(config_path)
    if not cfg["output"].get("save_embeddings", True):
        raise ValueError("Eksperimen 1A mewajibkan output.save_embeddings=true agar hasil dapat dianalisis ulang")
    seed = int(args.seed if args.seed is not None else cfg["experiment"].get("seed", 42))
    evaluation_seed = int(cfg["experiment"].get("evaluation_seed", seed + 1000))
    cfg["experiment"]["seed"] = seed
    seed_everything(seed, cfg["reproducibility"].get("deterministic", True), cfg["reproducibility"].get("warn_only", True))
    device = torch.device(args.device or ("cuda" if torch.cuda.is_available() else "cpu"))
    source_dir = Path(__file__).resolve().parent
    config_hash = stable_hash({"config": cfg, "smoke": args.smoke})
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = args.run_id or f"exp21d_v4_fairctrl_1a_seed{seed}_{config_hash}_{timestamp}"
    result_root = Path(cfg["output"]["result_dir"])
    if not result_root.is_absolute():
        result_root = (config_path.parent / result_root).resolve()
    run_dir = result_root / ("smoke" if args.smoke else "runs") / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    # Expose the path to the CLI-level failure handler before any expensive work.
    args._active_run_dir = run_dir
    logger = RunLogger(run_dir / "events.jsonl")
    manifest = {
        "experiment_id": "EXP21D-v4-FairCtrl-1A",
        "run_id": run_id,
        "status": "running",
        "scientific_status": "smoke_only_not_for_paper" if args.smoke else "paper_candidate_single_seed",
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "device": str(device),
        "config_hash": config_hash,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "git": git_revision(source_dir),
    }
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_started", run_id=run_id, device=str(device), smoke=args.smoke)
    if cfg["output"].get("snapshot_source", True):
        snapshot_source(source_dir, run_dir / "code_snapshot")
    (run_dir / "resolved_config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")
    write_json(run_dir / "environment.json", environment_manifest())

    started = time.time()
    dataset_path = Path(cfg["data"]["transactions"])
    if not dataset_path.is_absolute():
        dataset_path = (config_path.parent / dataset_path).resolve()
    manifest["dataset"] = file_fingerprint(dataset_path, cfg["reproducibility"].get("compute_dataset_sha256", False))
    write_json(run_dir / "manifest.json", manifest)
    logger.log("data_loading_started", dataset=str(dataset_path))
    time_cfg = cfg["time_window"]
    train, validation, test, gap, statistics = load_and_split_raw_data(
        str(dataset_path),
        cfg["data"].get("start_year", 2016),
        cfg["data"].get("max_rows"),
        time_cfg["reference_date"],
        int(time_cfg["train_days"]),
        int(time_cfg["val_days"]),
        int(time_cfg["gap_days"]),
        int(time_cfg["test_days"]),
    )
    if args.smoke:
        limit = int(cfg["smoke"].get("max_rows_per_split", 20_000))
        train = smoke_subset(train, limit, seed)
        validation = smoke_subset(validation, limit, seed + 1)
        test = smoke_subset(test, limit, seed + 2)
        gap = smoke_subset(gap, min(limit, len(gap)), seed + 3)
    feature_cfg = cfg["features"]
    train, validation, test, gap, feature_names, preprocessing = apply_strict_anti_leakage_preprocessing(
        train, validation, test, gap,
        feature_cfg["mode"], feature_cfg["use_cyclical_time"], feature_cfg["use_amount_log"],
    )
    expected = int(cfg["model"].get("input_dim_expected", 10))
    if len(feature_names) != expected:
        raise RuntimeError(f"Kontrak fitur gagal: diharapkan {expected}, aktual {len(feature_names)} ({feature_names})")
    save_preprocessing_artifacts(run_dir / "preprocessing_artifacts.json", preprocessing)
    write_json(run_dir / "window_statistics.json", statistics)
    logger.log("data_prepared", features=feature_names, rows={"train": len(train), "validation": len(validation), "test": len(test)})

    arrays: dict[str, np.ndarray] = {}
    labels: dict[str, np.ndarray] = {}
    ids: dict[str, np.ndarray] = {}
    frames = {"train": train, "validation": validation, "test": test}
    for split, frame in frames.items():
        arrays[split] = np.ascontiguousarray(frame[feature_names].to_numpy(dtype=np.float32))
        labels[split] = frame["Is Fraud?"].to_numpy(dtype=np.float32)
        ids[split] = frame["transaction_id"].to_numpy(dtype=np.int64)
    del train, validation, test, gap, frames
    gc.collect()

    training_cfg = cfg["training"]
    train_loader = make_loader(arrays["train"], labels["train"], int(training_cfg["batch_size"]), True, seed)
    val_loader = make_loader(arrays["validation"], labels["validation"], int(training_cfg["inference_batch_size"]), False, evaluation_seed)
    model = TabularEmbeddingMLP(len(feature_names), int(cfg["model"]["hidden_channels"]), float(cfg["model"]["dropout"])).to(device)
    optimizer = optim.Adam(model.parameters(), lr=float(training_cfg["learning_rate"]), weight_decay=float(training_cfg["weight_decay"]))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=float(training_cfg["lr_scheduler_factor"]), patience=int(training_cfg["lr_scheduler_patience"])
    ) if training_cfg.get("use_lr_scheduler", True) else None
    positives = int(labels["train"].sum())
    negatives = int(len(labels["train"]) - positives)
    class_ratio = negatives / positives if positives else 1.0
    positive_weight = min(np.sqrt(class_ratio), float(training_cfg.get("max_pos_weight", 50.0)))
    epochs = int(cfg["smoke"]["epochs"] if args.smoke else training_cfg["epochs"])
    history: list[dict[str, Any]] = []
    best_auprc, best_epoch, patience_counter = float("-inf"), 0, 0
    checkpoint_path = run_dir / "models" / "best_mlp_checkpoint.pt"
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    logger.log("mlp_training_started", epochs=epochs, positive_weight=float(positive_weight))
    training_started = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for feature_batch, target_batch in train_loader:
            feature_batch, target_batch = feature_batch.to(device), target_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(feature_batch)
            loss = weighted_bce(logits, target_batch, positive_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(training_cfg["grad_clip_norm"]))
            optimizer.step()
            loss_sum += float(loss.item()) * len(target_batch)
            correct += int(((torch.sigmoid(logits) >= 0.5) == target_batch.bool()).sum().item())
            seen += len(target_batch)
        val_output = infer_head(model, val_loader, device)
        val_auprc = float(average_precision_score(val_output["labels"], val_output["probabilities"]))
        val_f1 = float(f1_score(val_output["labels"], val_output["probabilities"] >= 0.5, zero_division=0))
        if scheduler:
            scheduler.step(val_auprc)
        record = {
            "epoch": epoch,
            "train_loss": loss_sum / max(seen, 1),
            "train_accuracy": correct / max(seen, 1),
            "val_loss": val_output["loss"],
            "val_auprc": val_auprc,
            "val_f1_at_05": val_f1,
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "elapsed_seconds": time.time() - training_started,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(run_dir / "training_history.csv", index=False)
        write_json(run_dir / "training_history.json", history)
        logger.log("epoch_completed", **record)
        print(f"Epoch {epoch:02d}/{epochs:02d} train_loss={record['train_loss']:.6f} val_auprc={val_auprc:.6f} val_f1={val_f1:.6f}")
        if val_auprc > best_auprc:
            best_auprc, best_epoch, patience_counter = val_auprc, epoch, 0
            torch.save({"state_dict": model.state_dict(), "epoch": epoch, "val_auprc": val_auprc, "feature_names": feature_names, "config": cfg}, checkpoint_path)
            logger.log("checkpoint_selected", epoch=epoch, val_auprc=val_auprc)
        else:
            patience_counter += 1
            if patience_counter >= int(training_cfg["early_stopping_patience"]):
                logger.log("early_stopping", epoch=epoch)
                break
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    training_seconds = time.time() - training_started

    feature_dir = run_dir / "features"
    feature_paths: dict[str, dict[str, Path]] = {}
    for split in ("train", "validation", "test"):
        feature_paths[split] = extract_embeddings(
            model, arrays[split], labels[split], ids[split], feature_dir, split,
            int(training_cfg["inference_batch_size"]), device, int(cfg["model"]["hidden_channels"]),
            bool(cfg["output"].get("save_tabular_features", True)),
        )
        feature_paths[split]["hybrid"] = assemble_hybrid(
            arrays[split], feature_paths[split]["embedding"], feature_dir / f"{split}_hybrid_features.npy"
        )
    logger.log("embeddings_saved", directory=str(feature_dir))

    direct_val = infer_head(model, val_loader, device)
    test_loader = make_loader(arrays["test"], labels["test"], int(training_cfg["inference_batch_size"]), False, evaluation_seed + 1)
    direct_test = infer_head(model, test_loader, device)
    min_precision = float(cfg["threshold_calibration"]["min_precision_constraint"])
    direct_threshold, direct_calibration = find_optimal_threshold(direct_val["probabilities"], direct_val["labels"], min_precision)
    direct_metrics = evaluate_predictions(direct_test["probabilities"], direct_test["labels"], direct_threshold)

    features_by_model = {
        "tabular_10d": {split: arrays[split] for split in arrays},
        "mlp_embedding_128d": {split: np.load(feature_paths[split]["embedding"], mmap_mode="r") for split in arrays},
        "hybrid_tabular_mlp_138d": {split: np.load(feature_paths[split]["hybrid"], mmap_mode="r") for split in arrays},
    }
    parameters = xgb_parameters(cfg["classifier"], class_ratio, seed, args.smoke)
    model_results: dict[str, Any] = {}
    xgboost_histories: dict[str, Any] = {}
    saved_probabilities: dict[str, np.ndarray] = {
        "validation_labels": labels["validation"].astype(np.int8),
        "test_labels": labels["test"].astype(np.int8),
        "validation_transaction_ids": ids["validation"],
        "test_transaction_ids": ids["test"],
        "direct_mlp_validation_probabilities": direct_val["probabilities"].astype(np.float32),
        "direct_mlp_test_probabilities": direct_test["probabilities"].astype(np.float32),
    }
    for key, display_name in MODEL_SPECS.items():
        logger.log("xgboost_training_started", model=key, dimensions=int(features_by_model[key]["train"].shape[1]))
        stage_start = time.time()
        classifier = XGBClassifier(**parameters)
        classifier.fit(
            features_by_model[key]["train"], labels["train"],
            eval_set=[(features_by_model[key]["validation"], labels["validation"])], verbose=False,
        )
        xgboost_histories[key] = classifier.evals_result()
        write_json(run_dir / "xgboost_training_history.json", xgboost_histories)
        classifier.save_model(str(run_dir / "models" / f"{key}_xgboost.json"))
        val_probability = classifier.predict_proba(features_by_model[key]["validation"])[:, 1]
        threshold, calibration = find_optimal_threshold(val_probability, labels["validation"], min_precision)
        test_probability = classifier.predict_proba(features_by_model[key]["test"])[:, 1]
        evaluation = evaluate_predictions(test_probability, labels["test"], threshold)
        model_results[key] = {
            "display_name": display_name,
            "dimensions": int(features_by_model[key]["train"].shape[1]),
            "threshold_calibration": calibration,
            **evaluation,
            "threshold_curve": threshold_curve(test_probability, labels["test"]),
            "runtime_seconds": time.time() - stage_start,
        }
        saved_probabilities[f"{key}_validation_probabilities"] = val_probability.astype(np.float32)
        saved_probabilities[f"{key}_test_probabilities"] = test_probability.astype(np.float32)
        logger.log("xgboost_training_completed", model=key, auprc=evaluation["metrics_threshold_independent"]["auprc"])
        del classifier
        gc.collect()
    np.savez_compressed(run_dir / "predictions.npz", **saved_probabilities)

    baseline_dir = Path(cfg["baseline"]["exp21d_v1_result_dir"])
    if not baseline_dir.is_absolute():
        baseline_dir = (config_path.parent / baseline_dir).resolve()
    baseline_rows = load_graphsage_baselines(baseline_dir, list(cfg["baseline"]["samplers"]))
    mlp_metrics = model_results["mlp_embedding_128d"]
    fair_rows = [{
        "representation": "Tabular MLP embedding",
        "sampler": "none",
        "dimensions": 128,
        "downstream_classifier": "XGBoost",
        "test_auprc": mlp_metrics["metrics_threshold_independent"]["auprc"],
        "test_roc_auc": mlp_metrics["metrics_threshold_independent"]["roc_auc"],
        "test_f1": mlp_metrics["metrics_optimal_threshold"]["f1_score"],
        "test_recall": mlp_metrics["metrics_optimal_threshold"]["recall"],
        "test_precision": mlp_metrics["metrics_optimal_threshold"]["precision"],
        "optimal_threshold": mlp_metrics["metrics_optimal_threshold"]["threshold"],
        "source": str((run_dir / "metrics.json").resolve()),
    }, *baseline_rows]
    fair_frame = pd.DataFrame(fair_rows)
    fair_frame.to_csv(run_dir / "fair_128d_comparison.csv", index=False)
    internal_rows = []
    for key, value in model_results.items():
        internal_rows.append({
            "model": key,
            "display_name": value["display_name"],
            "dimensions": value["dimensions"],
            "test_auprc": value["metrics_threshold_independent"]["auprc"],
            "test_roc_auc": value["metrics_threshold_independent"]["roc_auc"],
            "test_f1": value["metrics_optimal_threshold"]["f1_score"],
            "test_recall": value["metrics_optimal_threshold"]["recall"],
            "test_precision": value["metrics_optimal_threshold"]["precision"],
            "optimal_threshold": value["metrics_optimal_threshold"]["threshold"],
        })
    pd.DataFrame(internal_rows).to_csv(run_dir / "mlp_feature_sets_comparison.csv", index=False)

    statistics["frozen_test_evaluation"] = {"rows": len(labels["test"]), "fraud": int(labels["test"].sum()), "evaluated_once": True}
    write_json(run_dir / "window_statistics.json", statistics)
    output = {
        "experiment_id": "EXP21D-v4-FairCtrl-1A",
        "run_id": run_id,
        "scientific_status": manifest["scientific_status"],
        "seed": seed,
        "feature_names": feature_names,
        "model_architecture": {"input_dimensions": len(feature_names), "embedding_dimensions": int(cfg["model"]["hidden_channels"]), "hidden_layers": 2},
        "training": {"best_epoch": best_epoch, "best_validation_auprc": best_auprc, "positive_weight": float(positive_weight), "class_ratio": float(class_ratio)},
        "xgboost_parameters": parameters,
        "direct_mlp_head": {"threshold_calibration": direct_calibration, **direct_metrics},
        "models": model_results,
        "xgboost_training_history_file": "xgboost_training_history.json",
        "graphsage_baseline_source": str(baseline_dir),
        "runtime_seconds": {"mlp_training": training_seconds, "total": time.time() - started},
        "reanalysis_contract": {
            "predictions": "predictions.npz contains aligned validation/test labels, IDs, and probabilities",
            "features": "features/*.npy contains mmap-compatible tabular, 128-d embedding, and 138-d hybrid matrices",
            "history": "training_history.csv and events.jsonl preserve epoch and run history",
        },
    }
    write_json(run_dir / "metrics.json", output)
    manifest.update({"status": "complete", "completed_at_utc": datetime.now(timezone.utc).isoformat(), "best_epoch": best_epoch, "test_evaluations": 1})
    write_json(run_dir / "manifest.json", manifest)
    logger.log("run_completed", total_seconds=time.time() - started)
    build_artifact_index(run_dir)
    return run_dir


def main() -> None:
    parser = argparse.ArgumentParser(description="EXP21D-v4 Fair Control 1A: 128-d MLP vs 128-d GraphSAGE embeddings")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", default=None)
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--run-id", default=None)
    args = parser.parse_args()
    run_dir: Path | None = None
    try:
        run_dir = run_experiment(args)
        print(json.dumps({"status": "complete", "run_dir": str(run_dir)}, indent=2))
    except Exception as error:
        run_dir = run_dir or getattr(args, "_active_run_dir", None)
        if run_dir is not None:
            manifest_path = run_dir / "manifest.json"
            manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}
            manifest.update({"status": "failed", "error": repr(error), "traceback": traceback.format_exc()})
            write_json(manifest_path, manifest)
            RunLogger(run_dir / "events.jsonl").log("run_failed", error=repr(error))
            build_artifact_index(run_dir)
        raise


if __name__ == "__main__":
    main()


In [ ]:
%%writefile visualize_results.py
from __future__ import annotations

import argparse
import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.metrics import average_precision_score, precision_recall_curve

from artifacts import build_artifact_index


COLORS = {
    "tabular_10d": "#E69F00",
    "mlp_embedding_128d": "#0072B2",
    "hybrid_tabular_mlp_138d": "#009E73",
    "direct_mlp": "#CC79A7",
}


def save_figure(figure: plt.Figure, output_dir: Path, name: str, dpi: int) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        figure.savefig(output_dir / f"{name}.{suffix}", dpi=dpi, bbox_inches="tight")
    plt.close(figure)


def plot_fair_comparison(run_dir: Path, output_dir: Path, dpi: int) -> None:
    frame = pd.read_csv(run_dir / "fair_128d_comparison.csv")
    labels = frame["representation"].str.replace(" embedding", "", regex=False)
    figure, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    palette = ["#0072B2", "#56B4E9", "#009E73", "#D55E00"][: len(frame)]
    for axis, column, title in zip(axes, ["test_auprc", "test_f1"], ["Test AUPRC", "Test F1 at validation-selected threshold"]):
        bars = axis.bar(labels, frame[column], color=palette, edgecolor="black", linewidth=0.7)
        axis.set_title(title, fontweight="bold")
        axis.set_ylim(max(0, frame[column].min() - 0.06), min(1.0, frame[column].max() + 0.04))
        axis.tick_params(axis="x", rotation=22)
        axis.grid(axis="y", linestyle="--", alpha=0.35)
        axis.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
    figure.suptitle("Fair 128-d supervised neural embedding comparison (same XGBoost)", fontweight="bold")
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_fair_128d_embedding_comparison", dpi)


def plot_internal_comparison(run_dir: Path, output_dir: Path, dpi: int) -> None:
    frame = pd.read_csv(run_dir / "mlp_feature_sets_comparison.csv")
    x = np.arange(len(frame))
    width = 0.36
    figure, axis = plt.subplots(figsize=(10, 5.5))
    f1 = axis.bar(x - width / 2, frame["test_f1"], width, label="F1", color="#0072B2")
    auprc = axis.bar(x + width / 2, frame["test_auprc"], width, label="AUPRC", color="#009E73")
    axis.set_xticks(x, ["Tabular 10-d", "MLP Emb. 128-d", "Hybrid 138-d"])
    axis.set_ylim(max(0, min(frame["test_f1"].min(), frame["test_auprc"].min()) - 0.08), 1.0)
    axis.set_title("EXP21D-v4 FairCtrl-1A feature-set ablation", fontweight="bold")
    axis.grid(axis="y", linestyle="--", alpha=0.35)
    axis.legend()
    axis.bar_label(f1, fmt="%.4f", padding=3, fontsize=9)
    axis.bar_label(auprc, fmt="%.4f", padding=3, fontsize=9)
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_mlp_feature_set_ablation", dpi)


def plot_learning_curve(run_dir: Path, output_dir: Path, dpi: int) -> None:
    frame = pd.read_csv(run_dir / "training_history.csv")
    figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(frame["epoch"], frame["train_loss"], marker="o", label="Train weighted BCE")
    axes[0].plot(frame["epoch"], frame["val_loss"], marker="s", label="Validation BCE")
    axes[0].set_title("MLP loss history", fontweight="bold")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(frame["epoch"], frame["val_auprc"], marker="o", label="Validation AUPRC")
    axes[1].plot(frame["epoch"], frame["val_f1_at_05"], marker="s", label="Validation F1 @ 0.5")
    axes[1].set_title("Validation history", fontweight="bold")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    for axis in axes:
        axis.grid(linestyle="--", alpha=0.35)
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_mlp_training_history", dpi)


def plot_pr_curves(run_dir: Path, output_dir: Path, dpi: int) -> None:
    data = np.load(run_dir / "predictions.npz")
    labels = data["test_labels"]
    figure, axis = plt.subplots(figsize=(8, 6))
    export = []
    for key, display in {
        "tabular_10d": "Tabular 10-d + XGBoost",
        "mlp_embedding_128d": "MLP embedding 128-d + XGBoost",
        "hybrid_tabular_mlp_138d": "Hybrid MLP 138-d + XGBoost",
        "direct_mlp": "Direct MLP head",
    }.items():
        probabilities = data[f"{key}_test_probabilities"]
        precision, recall, _ = precision_recall_curve(labels, probabilities)
        area = average_precision_score(labels, probabilities)
        axis.plot(recall, precision, linewidth=2, color=COLORS[key], label=f"{display} (AUPRC≈{area:.4f})")
        stride = max(1, len(precision) // 2000)
        export.extend({"model": key, "recall": float(r), "precision": float(p)} for r, p in zip(recall[::stride], precision[::stride]))
    pd.DataFrame(export).to_csv(output_dir / "data_pr_curves.csv", index=False)
    axis.axhline(labels.mean(), color="gray", linestyle="--", label=f"Fraud prevalence ({labels.mean():.4%})")
    axis.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1.01), ylim=(0, 1.02))
    axis.set_title("Precision–Recall curves from persisted test probabilities", fontweight="bold")
    axis.grid(linestyle="--", alpha=0.35)
    axis.legend(loc="lower left", fontsize=9)
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_pr_curves_mlp_control", dpi)


def plot_threshold_sensitivity(run_dir: Path, output_dir: Path, dpi: int) -> None:
    metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    figure, axis = plt.subplots(figsize=(9, 5.5))
    rows = []
    for key, result in metrics["models"].items():
        curve = result["threshold_curve"]
        axis.plot(curve["threshold"], curve["f1"], label=result["display_name"], color=COLORS[key], linewidth=2)
        rows.extend({"model": key, "threshold": threshold, "f1": f1} for threshold, f1 in zip(curve["threshold"], curve["f1"]))
    pd.DataFrame(rows).to_csv(output_dir / "data_threshold_sensitivity.csv", index=False)
    axis.set(xlabel="Decision threshold", ylabel="Test F1", xlim=(0.05, 0.95), ylim=(0, 1.0))
    axis.set_title("Threshold sensitivity (diagnostic; threshold selected on validation)", fontweight="bold")
    axis.grid(linestyle="--", alpha=0.35)
    axis.legend(fontsize=9)
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_threshold_sensitivity", dpi)


def plot_confusion_matrices(run_dir: Path, output_dir: Path, dpi: int) -> None:
    metrics = json.loads((run_dir / "metrics.json").read_text(encoding="utf-8"))
    figure, axes = plt.subplots(1, 3, figsize=(13, 4.2))
    for axis, (key, result) in zip(axes, metrics["models"].items()):
        cm = result["metrics_optimal_threshold"]["confusion_matrix"]
        matrix = np.array([[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]])
        axis.imshow(matrix, cmap="Blues")
        for row in range(2):
            for column in range(2):
                axis.text(column, row, f"{matrix[row, column]:,}", ha="center", va="center", color="white" if matrix[row, column] > matrix.max() / 2 else "black")
        axis.set_xticks([0, 1], ["Pred 0", "Pred 1"])
        axis.set_yticks([0, 1], ["True 0", "True 1"])
        axis.set_title(key.replace("_", "\n"), fontsize=10, fontweight="bold")
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_confusion_matrices", dpi)


def plot_tsne(run_dir: Path, output_dir: Path, dpi: int, max_points: int, seed: int) -> None:
    embeddings = np.load(run_dir / "features" / "test_mlp_embeddings.npy", mmap_mode="r")
    labels = np.load(run_dir / "features" / "test_labels.npy", mmap_mode="r")
    rng = np.random.default_rng(seed)
    positive = np.flatnonzero(labels == 1)
    negative = np.flatnonzero(labels == 0)
    positive_take = min(len(positive), max_points // 2)
    negative_take = min(len(negative), max_points - positive_take)
    selected = np.concatenate([
        rng.choice(positive, positive_take, replace=False),
        rng.choice(negative, negative_take, replace=False),
    ])
    rng.shuffle(selected)
    projection = TSNE(n_components=2, init="pca", learning_rate="auto", perplexity=min(30, max(5, len(selected) // 100)), random_state=seed).fit_transform(np.asarray(embeddings[selected]))
    data = pd.DataFrame({"tsne_1": projection[:, 0], "tsne_2": projection[:, 1], "label": np.asarray(labels[selected], dtype=int)})
    data.to_csv(output_dir / "data_tsne_mlp_embedding.csv", index=False)
    figure, axis = plt.subplots(figsize=(8, 6))
    normal = data["label"] == 0
    axis.scatter(data.loc[normal, "tsne_1"], data.loc[normal, "tsne_2"], s=7, alpha=0.25, color="#56B4E9", label="Non-fraud")
    axis.scatter(data.loc[~normal, "tsne_1"], data.loc[~normal, "tsne_2"], s=14, alpha=0.8, color="#D55E00", label="Fraud")
    axis.set_title("t-SNE of persisted 128-d supervised MLP embeddings", fontweight="bold")
    axis.set_xlabel("t-SNE 1")
    axis.set_ylabel("t-SNE 2")
    axis.legend()
    figure.tight_layout()
    save_figure(figure, output_dir, "fig_tsne_mlp_embedding", dpi)


def main() -> None:
    parser = argparse.ArgumentParser(description="Generate paper-ready visuals without retraining")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--dpi", type=int, default=300)
    parser.add_argument("--tsne-max-points", type=int, default=5000)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--skip-tsne", action="store_true")
    args = parser.parse_args()
    run_dir = Path(args.run_dir)
    output_dir = run_dir / "figures"
    output_dir.mkdir(parents=True, exist_ok=True)
    plot_fair_comparison(run_dir, output_dir, args.dpi)
    plot_internal_comparison(run_dir, output_dir, args.dpi)
    plot_learning_curve(run_dir, output_dir, args.dpi)
    plot_pr_curves(run_dir, output_dir, args.dpi)
    plot_threshold_sensitivity(run_dir, output_dir, args.dpi)
    plot_confusion_matrices(run_dir, output_dir, args.dpi)
    if not args.skip_tsne:
        plot_tsne(run_dir, output_dir, args.dpi, args.tsne_max_points, args.seed)
    build_artifact_index(run_dir)
    print(json.dumps({"status": "complete", "figure_dir": str(output_dir)}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile reanalyze.py
from __future__ import annotations

import argparse
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from artifacts import build_artifact_index, write_json
from metrics import evaluate_predictions, find_optimal_threshold, threshold_curve


MODEL_KEYS = ("tabular_10d", "mlp_embedding_128d", "hybrid_tabular_mlp_138d", "direct_mlp")


def main() -> None:
    parser = argparse.ArgumentParser(description="Recompute metrics from persisted probabilities; no model training")
    parser.add_argument("--run-dir", required=True)
    parser.add_argument("--min-precision", type=float, default=0.15)
    parser.add_argument("--tag", default=None, help="Optional output tag, e.g. recall_at_200")
    args = parser.parse_args()
    run_dir = Path(args.run_dir)
    predictions = np.load(run_dir / "predictions.npz")
    validation_labels = predictions["validation_labels"]
    test_labels = predictions["test_labels"]
    result = {
        "source_run": str(run_dir.resolve()),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "min_precision_constraint": args.min_precision,
        "training_performed": False,
        "models": {},
    }
    for key in MODEL_KEYS:
        validation = predictions[f"{key}_validation_probabilities"]
        test = predictions[f"{key}_test_probabilities"]
        threshold, calibration = find_optimal_threshold(validation, validation_labels, args.min_precision)
        result["models"][key] = {
            "threshold_calibration": calibration,
            **evaluate_predictions(test, test_labels, threshold),
            "threshold_curve": threshold_curve(test, test_labels),
        }
    tag = args.tag or f"min_precision_{str(args.min_precision).replace('.', 'p')}"
    output = run_dir / "reanalysis" / f"metrics_{tag}.json"
    write_json(output, result)
    build_artifact_index(run_dir)
    print(json.dumps({"status": "complete", "output": str(output)}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run.py
from __future__ import annotations

import argparse
import subprocess
import sys
from pathlib import Path

import yaml


def newest_run(root: Path, smoke: bool) -> Path:
    parent = root / ("smoke" if smoke else "runs")
    candidates = [item for item in parent.glob("exp21d_v4_fairctrl_1a_*") if item.is_dir()]
    if not candidates:
        raise RuntimeError(f"Tidak ada run baru di {parent}")
    return max(candidates, key=lambda item: item.stat().st_mtime_ns)


def resolve_kaggle_paths(config_path: Path) -> None:
    if not Path("/kaggle/input").exists():
        return
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    dataset = sorted(Path("/kaggle/input").rglob("credit_card_transactions-ibm_v2.csv"))
    if not dataset:
        raise FileNotFoundError("Dataset IBM tidak ditemukan di /kaggle/input")
    config["data"]["transactions"] = str(dataset[0])
    baseline_candidates = [path.parent for path in Path("/kaggle/input").rglob("exp21d_uniform_results.json")]
    if not baseline_candidates:
        raise FileNotFoundError("Artefak EXP21D v1 tidak ditemukan di /kaggle/input")
    config["baseline"]["exp21d_v1_result_dir"] = str(baseline_candidates[0])
    config_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="Run EXP21D-v4 FairCtrl-1A and all visualizations")
    parser.add_argument("--config", default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", default=None)
    parser.add_argument("--smoke", action="store_true")
    parser.add_argument("--skip-tsne", action="store_true")
    args = parser.parse_args()
    source = Path(__file__).resolve().parent
    config_path = Path(args.config) if args.config else source / ("config.kaggle.yaml" if Path("/kaggle/input").exists() else "config.reference.yaml")
    config_path = config_path.resolve()
    resolve_kaggle_paths(config_path)
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    result_root = Path(config["output"]["result_dir"])
    if not result_root.is_absolute():
        result_root = (config_path.parent / result_root).resolve()
    command = [sys.executable, str(source / "main.py"), "--config", str(config_path)]
    if args.seed is not None:
        command += ["--seed", str(args.seed)]
    if args.device:
        command += ["--device", args.device]
    if args.smoke:
        command.append("--smoke")
    subprocess.run(command, cwd=source, check=True)
    run_dir = newest_run(result_root, args.smoke)
    visual = [sys.executable, str(source / "visualize_results.py"), "--run-dir", str(run_dir)]
    if args.skip_tsne:
        visual.append("--skip-tsne")
    subprocess.run(visual, cwd=source, check=True)
    print(f"Run dan visualisasi selesai: {run_dir}")


if __name__ == "__main__":
    main()


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
!python run.py --config config.kaggle.yaml
